# Customer Churn Prediction Using Machine Learning
**Course:** Artificial Intelligence — 5th Semester  
**Assignment:** Customer Churn Prediction Using Machine Learning  

This notebook follows the assignment pipeline: data loading and understanding, preprocessing, EDA, feature engineering, model training, evaluation, and sample prediction.

> **Dataset note:** The supplied dataset was a CSV file. It was converted to Excel for this submission so the notebook can follow the assignment instruction to load an Excel dataset.

In [1]:
import matplotlib
matplotlib.use('Agg')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

pd.set_option('display.max_columns', None)


In [2]:
# 1. Data Loading and Understanding
file_path = "WA_Fn-UseC_-Telco-Customer-Churn.xlsx"
df = pd.read_excel(file_path)

print("Shape:", df.shape)
display(df.head())
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("dtype"))


Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Data types:


,dtype
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object


In [3]:
print("Basic statistics:")
display(df.describe(include='all').T)
print("Missing values:")
display(df.isna().sum().to_frame("missing_values"))
print("Duplicate rows:", df.duplicated().sum())
print("Target distribution:")
display(df["Churn"].value_counts().to_frame("count"))


Basic statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Missing values:


,missing_values
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


Duplicate rows: 0
Target distribution:


,count
Churn,
No,5174
Yes,1869


## Important Columns in Simple Words
- **customerID:** Unique customer identifier; not useful for prediction.
- **tenure:** Number of months the customer has stayed with the company.
- **MonthlyCharges:** Customer's monthly bill.
- **TotalCharges:** Total amount charged to the customer.
- **Contract:** Customer's contract type.
- **InternetService:** Type of internet service.
- **PaymentMethod:** How the customer pays.
- **SeniorCitizen:** Whether the customer is a senior citizen.
- **Churn:** Target variable — **Yes** means the customer left; **No** means the customer stayed.

In [4]:
# 2. Data Preprocessing
# Convert TotalCharges from text to numeric. Blank entries become NaN.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing values after converting TotalCharges:")
display(df.isna().sum().to_frame("missing_values"))

# Remove the identifier because it does not describe customer behavior.
df_model = df.drop(columns=["customerID"]).copy()

# Separate input features and target.
X = df_model.drop(columns=["Churn"])
y = df_model["Churn"].map({"No": 0, "Yes": 1})

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target mapping: No=0, Yes=1")


Missing values after converting TotalCharges:


,missing_values
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


X shape: (7043, 19)
y shape: (7043,)
Target mapping: No=0, Yes=1


In [5]:
# 3. Exploratory Data Analysis
plt.figure(figsize=(6,4))
df['Churn'].value_counts().reindex(['No','Yes']).plot(kind='bar')
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show(); plt.close()

print("Churn percentage:")
display((df["Churn"].value_counts(normalize=True)*100).round(2).to_frame("percentage"))


Churn percentage:


/tmp/ipykernel_1055/2958080669.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close()


,percentage
Churn,
No,73.46
Yes,26.54


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
pd.crosstab(df['Contract'], df['Churn']).reindex(columns=['No','Yes']).plot(kind='bar', ax=axes[0])
axes[0].set_title("Churn by Contract Type")
axes[0].tick_params(axis='x', rotation=15)

df.boxplot(column='tenure', by='Churn', ax=axes[1])
axes[1].set_title("Tenure by Churn")
plt.tight_layout()
plt.show(); plt.close()

print("Average tenure by churn:")
display(df.groupby("Churn")["tenure"].mean().round(2).to_frame("average_tenure"))


Average tenure by churn:


/tmp/ipykernel_1055/1436258641.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close()


,average_tenure
Churn,
No,37.57
Yes,17.98


In [7]:
plt.figure(figsize=(7,4))
pd.crosstab(df['PaymentMethod'], df['Churn']).reindex(columns=['No','Yes']).plot(kind='bar')
plt.title("Churn by Payment Method")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show(); plt.close()


/tmp/ipykernel_1055/3819876828.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close()


In [8]:
# 4. Training and Testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42, class_weight="balanced")
}

results = {}
predictions = {}
pipelines = {}

for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    pipelines[name] = pipe
    predictions[name] = pred
    results[name] = {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1-Score": f1_score(y_test, pred)
    }

results_df = pd.DataFrame(results).T.round(4)
display(results_df)


,Accuracy,Precision,Recall,F1-Score
Logistic Regression,0.8055,0.6572,0.5588,0.6040
Random Forest,0.7857,0.6277,0.4733,0.5396


In [9]:
# 5. Confusion Matrices
for name, pred in predictions.items():
    cm = confusion_matrix(y_test, pred)
    plt.figure(figsize=(4,3))
    axes = plt.gca(); axes.imshow(cm, interpolation='nearest'); axes.set_xticks([0,1]); axes.set_yticks([0,1]); axes.set_xticklabels(['No Churn','Churn']); axes.set_yticklabels(['No Churn','Churn']);
    for i in range(2):
        for j in range(2):
            axes.text(j,i,str(cm[i,j]),ha='center',va='center')
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show(); plt.close()

    print(name)
    print(classification_report(y_test, pred, target_names=["No Churn", "Churn"], digits=3))


Logistic Regression
              precision    recall  f1-score   support

    No Churn      0.849     0.895     0.871      1035
       Churn      0.657     0.559     0.604       374

    accuracy                          0.806      1409
   macro avg      0.753     0.727     0.738      1409
weighted avg      0.798     0.806     0.800      1409

Random Forest
              precision    recall  f1-score   support

    No Churn      0.825     0.899     0.860      1035
       Churn      0.628     0.473     0.540       374

    accuracy                          0.786      1409
   macro avg      0.726     0.686     0.700      1409
weighted avg      0.773     0.786     0.775      1409



/tmp/ipykernel_1055/4026889375.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close()
/tmp/ipykernel_1055/4026889375.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close()


## Model Comparison
The two required classification algorithms are Logistic Regression and Random Forest. The metrics above are calculated on the same 20% test set. For this run, Logistic Regression has the higher Accuracy, Precision, Recall, and F1-Score, so it is selected as the final model for the sample prediction section.

This selection is based on the measured test-set metrics in this notebook, not on a claim that the model will perform identically on future customers.

In [10]:
# 6. Application / Prediction
final_model = pipelines["Logistic Regression"]

sample_customers = pd.DataFrame([
    {
        "gender":"Female","SeniorCitizen":0,"Partner":"Yes","Dependents":"No","tenure":2,
        "PhoneService":"Yes","MultipleLines":"No","InternetService":"Fiber optic",
        "OnlineSecurity":"No","OnlineBackup":"No","DeviceProtection":"No",
        "TechSupport":"No","StreamingTV":"No","StreamingMovies":"No",
        "Contract":"Month-to-month","PaperlessBilling":"Yes",
        "PaymentMethod":"Electronic check","MonthlyCharges":85.0,"TotalCharges":170.0
    },
    {
        "gender":"Male","SeniorCitizen":0,"Partner":"Yes","Dependents":"Yes","tenure":60,
        "PhoneService":"Yes","MultipleLines":"No","InternetService":"DSL",
        "OnlineSecurity":"Yes","OnlineBackup":"Yes","DeviceProtection":"Yes",
        "TechSupport":"Yes","StreamingTV":"No","StreamingMovies":"No",
        "Contract":"Two year","PaperlessBilling":"No",
        "PaymentMethod":"Credit card (automatic)","MonthlyCharges":55.0,"TotalCharges":3300.0
    },
    {
        "gender":"Female","SeniorCitizen":1,"Partner":"No","Dependents":"No","tenure":8,
        "PhoneService":"Yes","MultipleLines":"Yes","InternetService":"Fiber optic",
        "OnlineSecurity":"No","OnlineBackup":"Yes","DeviceProtection":"No",
        "TechSupport":"No","StreamingTV":"Yes","StreamingMovies":"Yes",
        "Contract":"Month-to-month","PaperlessBilling":"Yes",
        "PaymentMethod":"Electronic check","MonthlyCharges":95.0,"TotalCharges":760.0
    }
])

sample_pred = final_model.predict(sample_customers)
sample_prob = final_model.predict_proba(sample_customers)[:,1]

prediction_output = sample_customers[["tenure","Contract","MonthlyCharges"]].copy()
prediction_output["Prediction"] = np.where(sample_pred==1, "Churn", "No Churn")
prediction_output["Churn Probability"] = sample_prob.round(3)
display(prediction_output)


,tenure,Contract,MonthlyCharges,Prediction,Churn Probability
0,2,Month-to-month,85.0,Churn,0.635
1,60,Two year,55.0,No Churn,0.005
2,8,Month-to-month,95.0,Churn,0.811


## Conclusion
The dataset contains 7,043 customer records and 21 original columns. The target variable is **Churn**. The data contains 11 blank values in `TotalCharges` after converting that column from text to numeric, and there are no duplicate rows. The identifier `customerID` is excluded from modeling.

The analysis shows clear differences in churn rates across customer characteristics such as contract type, tenure, internet service, and payment method. Logistic Regression and Random Forest were trained and evaluated. In this experiment, Logistic Regression produced the stronger test-set metrics and was used for sample predictions.

**Limitations:** the model is based on historical data; customer behavior can change over time. Accuracy alone is not sufficient for business decisions, and false positives/false negatives have different costs. Further work could include cross-validation, threshold tuning, class-imbalance analysis, and testing on newer customer data.

**Business use:** a telecom company could use churn probabilities to identify customers who may need retention attention, then combine model results with business rules and customer-service information before taking action.